In [28]:
from fractions import Fraction
from typing import List, Tuple, Union
import numpy as np
import pandas as pd
from IPython.display import display
from sympy import Matrix
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.precision', 12)
pd.set_option('display.width', 300)
pd.set_option('display.max_columns', None)

In [29]:
def input_matrix(filename, convert_fractions=False):
    """
    Reads a matrix from a text file and returns it as a NumPy array.
    Supports fractional entries if present (though your files use decimals).
    """
    matrix = []

    with open(filename, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue

            row = []
            for token in tokens:
                if '/' in token:
                    # convert fractions if any includes fraction sign
                    val = Fraction(token)
                    row.append(float(val) if convert_fractions else val)
                else:
                    # parse as float directly
                    row.append(float(token))
            
            matrix.append(row)
            
    dtype = float if convert_fractions else object
    return np.array(matrix, dtype=dtype)

In [30]:
def output_matrix(X: np.ndarray, precision: int = 4):
    X = np.array(X)
    if np.iscomplexobj(X):
        display(Matrix(np.round(X.real, precision) + 1j * np.round(X.imag, precision)))
    else:
        X_float = np.array(X, dtype=float)
        display(Matrix(np.round(X_float, precision)))

# Phương pháp luỹ thừa và xuống thang

### 1. Thuật toán tìm giá trị riêng trội

> Note:
> - Đối xứng xác định dương, xác định không âm: Chỉ có TH1
> - Đối xứng: TH1 và TH2
> - Tìm giá trị kỳ dị lớn nhất -> Tìm trị riêng của $A^TA$ -> Chỉ có TH1
> 
> **KHI ĐỀ HỎI TÌM GIÁ TRỊ KỲ DỊ & VECTOR KỲ DỊ BẰNG LŨY THỪA / XUỐNG THANG:**
> Đặt $M = A A^T$ (nếu $m \le n$, cỡ $m \times m$) hoặc $M = A^T A$ (nếu $m > n$, cỡ $n \times n$). Đặt $r = \min(m, n)$.
> 
> **1. Nếu đề yêu cầu tìm Giá trị kỳ dị lớn nhất ($\sigma_1$) và Vector kỳ dị trái/phải tương ứng:**
> - **Chép thuật toán:** Chỉ chép duy nhất **TH1 (Thuật toán 1)** áp dụng cho $M$. Thu được $\lambda_1, v_1$.
> - **Kết luận:** $\sigma_1 = \sqrt{\lambda_1}$.
>   - Nếu $M = A^T A$: $v_1$ là vector kỳ dị phải. Tính vector kỳ dị trái $u_1 = \dfrac{A v_1}{\sigma_1}$.
>   - Nếu $M = A A^T$: $v_1$ là vector kỳ dị trái. Tính vector kỳ dị phải $v_1^* = \dfrac{A^T u_1}{\sigma_1}$.
> 
> **2. Nếu đề yêu cầu nêu ý tưởng hoặc tìm các giá trị kỳ dị tiếp theo ($\sigma_2, \sigma_3, \dots$) bằng Xuống thang:**
> - **Bỏ qua Bước 1 & 2 của Thuật toán 2** (vì $M$ đối xứng nên $w_1 = v_1$ và $c = v_1^T v_1 = 1$).
> - **Chép công thức xuống thang rút gọn:** $M^{(2)} = M - \lambda_1 v_1 v_1^T$.
> - **Hướng lặp:** Áp dụng lại TH1 (Thuật toán 1) cho $M^{(2)}$ thu được $\lambda_2 \Rightarrow \sigma_2 = \sqrt{\lambda_2}$ và vector riêng $v_2$. (Tiếp tục lặp nếu cần $\sigma_3$).
> 
> **3. Nếu đề yêu cầu tìm TẤT CẢ giá trị kỳ dị và vector kỳ dị bằng Lũy thừa + Xuống thang:**
> - **Khởi tạo:** Đặt $M^{(1)} = M$.
> - **Chép thuật toán:** Chép lại **Thuật toán 3** (Thuật toán kết hợp) nhưng sửa trực tiếp nội dung các Bước 2.1, 2.2 và 3 trong bài thi như sau:
>   - **Tại Bước 2.1:** Ghi là: *"Áp dụng Thuật toán 1 (lấy mục TH1) cho $M^{(k)}$ thu được $\lambda_k, v_k$. Tính giá trị kỳ dị $\sigma_k = \sqrt{\lambda_k}$"*.
>   - **Tại Bước 2.2:** Ghi thẳng công thức xuống thang rút gọn: *"Gán $M^{(k+1)} = M^{(k)} - \lambda_k v_k v_k^T$"* (Không chép Thuật toán 2).
>   - **Tại Bước 3:** Sửa thành xuất vector kỳ dị:
>     - Nếu $M = A^T A$: Tập kỳ dị phải là $\{v_1, \dots, v_n\}$. Tính toàn bộ tập kỳ dị trái $u_k = \dfrac{A v_k}{\sigma_k}$.
>     - Nếu $M = A A^T$: Tập kỳ dị trái là $\{v_1, \dots, v_m\}$. Tính toàn bộ tập kỳ dị phải $v_k^* = \dfrac{A^T u_k}{\sigma_k}$.

**Input**: $A_{n \times n}$, $x^{(0)}$, $\varepsilon$, số lần lặp tối đa $N$.

**Output**: Tập các giá trị riêng trội và vector riêng tương ứng.

1. Khởi tạo $x^{(0)} = \dfrac{x^{(0)}}{\|x^{(0)}\|_2}$.

2. Xét lần lượt các trường hợp để tìm trị riêng trội:

    2.1. TH1 ($|\lambda_1| > |\lambda_2|$):
    
    Lặp $k$ từ $1$ đến $N$:

    - Tính $y^{(k)} = A x^{(k-1)}$.
    
    - Tìm vị trí $j$ thỏa mãn $|x^{(k-1)}_j| = \max_i |x^{(k-1)}_i|$. Tính $\lambda^{(k)} = \dfrac{y^{(k)}_j}{x^{(k-1)}_j}$.
    
    - Tính $x^{(k)} = \dfrac{y^{(k)}}{\|y^{(k)}\|_2}$.

    - Nếu $\|y^{(k)} - \lambda^{(k)} x^{(k-1)}\|_2 < \varepsilon \Rightarrow$ Gán $\lambda_1 = \lambda^{(k)}, v_1 = x^{(k)}$, thoát Bước 2.

    - Nếu hết vòng lặp không hội tụ $\Rightarrow$ Chuyển sang 2.2.

    2.2. TH2 ($|\lambda_1| = |\lambda_2| > |\lambda_3|$, $\lambda_1 = -\lambda_2$): 
    
    Lặp $k$ từ $1$ đến $N$:

    - Tính $y^{(2k)} = A^{2k} x^{(0)}$ và $y^{(2k+2)} = A^{2k+2} x^{(0)}$.

    - Tính $\theta^{(k)} = \dfrac{[y^{(2k+2)}]_{j}}{[y^{(2k)}]_{j}}$, với $j$ là vị trí thỏa mãn $|[y^{(2k)}]_{j}| = \max_i |[y^{(2k)}]_{i}|$.

    - Nếu $\theta^{(k)} \in \mathbb{R}$ và $\theta^{(k)} > 0$, thực hiện:

      $\lambda_1 = \sqrt{\theta^{(k)}}$
      
      $\lambda_2 = -\sqrt{\theta^{(k)}}$

      $y^{(2k+1)} = A^{2k+1} x^{(0)}$

      $v_1 = \lambda_1 y^{(2k)} + y^{(2k+1)}$, $\,$ $v_1 = \dfrac{v_1}{\|v_1\|_2}$
      
      $v_2 = \lambda_1 y^{(2k)} - y^{(2k+1)}$, $\,$ $v_2 = \dfrac{v_2}{\|v_2\|_2}$

      Nếu $\|A v_1 - \lambda_1 v_1\|_2 < \varepsilon$ và $\|A v_2 - \lambda_2 v_2\|_2 < \varepsilon \Rightarrow$ Thoát Bước 2.

    - Nếu hết vòng lặp không hội tụ $\Rightarrow$ Chuyển sang 2.3.

    2.3. TH3 ($|\lambda_1| = |\lambda_2| > |\lambda_3|$, $\lambda_1 = \overline{\lambda_2}$): 
    
    Lặp $m$ từ $1$ đến $N$:

    - Tính $y^{(m)} = A^m x^{(0)}$, $y^{(m+1)} = A^{m+1} x^{(0)}$, $y^{(m+2)} = A^{m+2} x^{(0)}$.

    - Chọn 2 hàng $r, s$. Tính $p^{(m)}, q^{(m)}$ từ hệ phương trình:
    $$\begin{cases}
      [y^{(m+2)}]_{r} - p^{(m)} [y^{(m+1)}]_{r} + q^{(m)} [y^{(m)}]_{r} = 0\\
      [y^{(m+2)}]_{s} - p^{(m)} [y^{(m+1)}]_{s} + q^{(m)} [y^{(m)}]_{s} = 0
    \end{cases}$$

    - Tính $\Delta = (p^{(m)})^2 - 4q^{(m)}$. Thực hiện:

      Tính $\lambda_1, \lambda_2$ là nghiệm của $t^2 - p^{(m)}t + q^{(m)} = 0$.

      Gán $v_1 = y^{(m+1)} - \lambda_2 y^{(m)}$, $\,$ $v_1 = \dfrac{v_1}{\|v_1\|_2}$
      
      Gán $v_2 = y^{(m+1)} - \lambda_1 y^{(m)}$, $\,$ $v_2 = \dfrac{v_2}{\|v_2\|_2}$

      Nếu $\|A v_1 - \lambda_1 v_1\|_2 < \varepsilon$ và $\|A v_2 - \lambda_2 v_2\|_2 < \varepsilon \Rightarrow$ Thoát Bước 2.

3. Kết luận các giá trị riêng và vector riêng thu được.

### 2. Thuật toán Xuống thang

**Input**: $A_{n \times n}$, $\lambda_1, v_1$, $y$.

**Output**: Ma trận xuống thang $B$.

1. Áp dụng Thuật toán 1 đối với ma trận $A^T$ và vector khởi tạo $y$, thu được vector riêng trái $w_1$.

2. Tính $c = w_1^T v_1$.

3. Nếu $c = 0 \Rightarrow$ Báo lỗi, dừng chương trình. Trái lại, kết luận $B = A - \lambda_1 v_1 \left(\dfrac{w_1}{c}\right)^T$.

### 3. Thuật toán kết hợp

**Input**: $A_{n \times n}$, $x^{(0)}$, $\varepsilon$.

**Output**: Tập toàn bộ trị riêng và vector riêng.

1. Khởi tạo $M = A$, $y = x^{(0)}$, $k = 1$.

2. Lặp cho đến khi $k > n$:

    2.1. Áp dụng Thuật toán 1 cho ma trận $M$. Thu được tập $S$ chứa các cặp $(\lambda, v)$.

    2.2. Lặp mỗi cặp $(\lambda, v) \in S$:

    - Gán $\lambda_k = \lambda$ và $v_k = v$.

    - Áp dụng Thuật toán 2 cho các tham số $(M, \lambda_k, v_k, y)$. Gán $M = B$.

    - Gán $k = k + 1$.

    2.3. Gán $y = x^{(0)}$.

3. Kết luận toàn bộ các cặp trị riêng và vector riêng thu được.

In [31]:
def deflate_once(A, lam, v, x0, tol, max_iter, norm_ord):
    pairs = power_method(A.T, x0, tol=tol, max_iter=max_iter, norm_ord=norm_ord, display_df=False)
    _, w = pairs[0] 
    
    c = np.vdot(np.conj(w), v)
    if abs(c) < 1e-12:
        raise ValueError("Left and right eigenvectors are nearly orthogonal; cannot deflate.")
    
    return A - lam * np.outer(v, w/c)

def compute_all_eigenpairs(A, x0, tol=1e-6, max_iter=1000, norm_ord=2, precision=4):
    n = A.shape[0]
    M = A.copy().astype(complex)
    eigs = []
    vecs = []
    
    while len(eigs) < n:
        print(f"\n{'='*40}")
        print(f"BƯỚC {len(eigs)+1}: Tìm trị riêng cho ma trận")
        output_matrix(M, precision)
        
        pairs = power_method(M, x0, tol, max_iter, norm_ord, precision, display_df=True)
        
        for lam, v in pairs:
            if len(eigs) >= n:
                break
            eigs.append(lam)
            vecs.append(v)
            if len(eigs) < n:
                M = deflate_once(M, lam, v, x0, tol, max_iter, norm_ord)
                
    pairs = sorted(zip(eigs, vecs), key=lambda x: -abs(x[0]))
    eigs_sorted, vecs_sorted = zip(*pairs)
    
    print(f"\n{'='*40}")
    print("KẾT QUẢ CUỐI CÙNG:")
    for i, (lam, v) in enumerate(zip(eigs_sorted, vecs_sorted)):
        if abs(lam.imag) < 1e-10:
            print(f"λ_{i+1} = {lam.real:.4f}")
        else:
            sign = "+" if lam.imag > 0 else "-"
            print(f"λ_{i+1} = {lam.real:.4f} {sign} {abs(lam.imag):.4f}j")
        print(f"v_{i+1}:")
        v_clean = np.where(np.abs(v.imag) < 1e-10, v.real, v)
        output_matrix(v_clean, precision)
        
    return list(eigs_sorted), list(vecs_sorted)


In [32]:
def power_method(A, x0, tol=1e-6, max_iter=1000, norm_ord=2, precision=4, display_df=True):
    def fmt(v):
        try:
            if abs(v.imag) < 1e-10:
                return f"{v.real:.{precision}f}"
            sign = "+" if v.imag > 0 else "-"
            return f"{v.real:.{precision}f}{sign}{abs(v.imag):.{precision}f}j"
        except AttributeError:
            return f"{v:.{precision}f}"

    n = A.shape[0]
    x0 = x0.astype(A.dtype)
    x = x0 / np.linalg.norm(x0, ord=norm_ord)

    # --- TH1 ---
    eigenvalue = complex(float('inf'), float('inf'))
    history_th1 = []
    
    for k in range(1, max_iter + 1):
        y = A @ x
        
        j = np.argmax(np.abs(x))
        if abs(x[j]) < 1e-12: break
        lambda_new = y[j] / x[j]
        
        norm_y = np.linalg.norm(y, ord=norm_ord)
        if norm_y == 0:
            break
            
        x_new = y / norm_y
        history_th1.append([k] + y.tolist() + [lambda_new] + x_new.tolist())
        
        if np.linalg.norm(y - lambda_new * x) < tol:
            if display_df:
                cols = ['k'] + [f'y_{i+1}' for i in range(n)] + ['lambda'] + [f'x_{i+1}' for i in range(n)]
                df = pd.DataFrame(history_th1, columns=cols)
                print("--- Rơi vào TH1 ---")
                print(df.to_string(index=False, float_format=fmt))
            return [(lambda_new, x_new)]
        
        x = x_new
        eigenvalue = lambda_new

    # --- TH2 ---
    y0 = x.astype(A.dtype)
    theta_old = 0.0
    history_th2 = []
    y_even = y0
    
    A2 = A @ A
    
    for k in range(1, max_iter + 1):
        if k > 1:
            y_even = A2 @ y_even
        y_even2 = A2 @ y_even
        
        j = np.argmax(np.abs(y_even))
        theta_new = y_even2[j] / y_even[j]
        
        y_odd = A @ y_even
        
        if np.isreal(theta_new) and theta_new.real >= 0:
            lambda1 = np.sqrt(theta_new.real)
            v1 = lambda1 * y_even + y_odd
            v2 = lambda1 * y_even - y_odd
            v1_norm = np.linalg.norm(v1, ord=norm_ord)
            v2_norm = np.linalg.norm(v2, ord=norm_ord)
            v1 = v1 / v1_norm if v1_norm != 0 else v1
            v2 = v2 / v2_norm if v2_norm != 0 else v2
        else:
            lambda1 = 0
            v1 = np.zeros_like(y_even)
            v2 = np.zeros_like(y_even)
            
        history_th2.append([k] + y_even.tolist() + [theta_new] + v1.tolist() + v2.tolist())
        
        if abs(theta_new - theta_old) < tol and np.isreal(theta_new) and theta_new.real > 0:
            if display_df:
                cols = ['k'] + [f'y2k_{i+1}' for i in range(n)] + ['theta'] + [f'v1_{i+1}' for i in range(n)] + [f'v2_{i+1}' for i in range(n)]
                df = pd.DataFrame(history_th2, columns=cols)
                print("--- Rơi vào TH2 ---")
                print(df.to_string(index=False, float_format=fmt))
            
            res1 = np.linalg.norm(A @ v1 - lambda1 * v1)
            res2 = np.linalg.norm(A @ v2 + lambda1 * v2)
            if res1 < tol and res2 < tol:
                return [(lambda1, v1), (-lambda1, v2)]
            
        theta_old = theta_new

    # --- TH3 ---
    p_old, q_old = 0.0, 0.0
    y_m = x.astype(A.dtype)
    y_m1 = A @ y_m
    y_m2 = A @ y_m1
    history_th3 = []
    
    for m in range(1, max_iter + 1):
        best_det = -1
        best_r, best_s = 0, 1
        for r in range(n):
            for s in range(r+1, n):
                det = abs(y_m1[r]*y_m[s] - y_m1[s]*y_m[r])
                if det > best_det:
                    best_det = det
                    best_r, best_s = r, s
                    
        r, s = best_r, best_s
        M_mat = np.array([
            [-y_m1[r], y_m[r]],
            [-y_m1[s], y_m[s]]
        ])
        b_vec = np.array([-y_m2[r], -y_m2[s]])
        
        try:
            p_new, q_new = np.linalg.solve(M_mat, b_vec)
        except np.linalg.LinAlgError:
            p_new, q_new = p_old, q_old
            
        history_th3.append([m] + y_m.tolist() + [p_new, q_new])
        
        if m > 1 and abs(p_new - p_old) < tol and abs(q_new - q_old) < tol:
            if display_df:
                cols = ['m'] + [f'ym_{i+1}' for i in range(n)] + ['p', 'q']
                df = pd.DataFrame(history_th3, columns=cols)
                print("--- Rơi vào TH3 ---")
                print(df.to_string(index=False, float_format=fmt))
                
            delta = p_new**2 - 4*q_new
            if np.isreal(delta) and delta.real < 0:
                sqrt_delta = np.sqrt(complex(delta))
                lam1 = (p_new + sqrt_delta) / 2
                lam2 = (p_new - sqrt_delta) / 2
            else:
                lam1 = (p_new + np.sqrt(delta)) / 2
                lam2 = (p_new - np.sqrt(delta)) / 2
                
            v1 = y_m1 - lam2 * y_m
            v2 = y_m1 - lam1 * y_m
            
            v1 = v1 / np.linalg.norm(v1)
            v2 = v2 / np.linalg.norm(v2)
            
            res1 = np.linalg.norm(A @ v1 - lam1 * v1)
            res2 = np.linalg.norm(A @ v2 - lam2 * v2)
            if res1 < tol and res2 < tol:
                return [(lam1, v1), (lam2, v2)]
            
        p_old, q_old = p_new, q_new
        
        y_m = y_m1
        y_m1 = y_m2
        y_m2 = A @ y_m2
        
    raise RuntimeError("Power Method failed to converge in all 3 cases.")


In [33]:
A = input_matrix('PWDF_input_A.txt', convert_fractions=False)
x0 = np.array([1, 1, 1, 1, 1], dtype=complex)
eigs, vecs = compute_all_eigenpairs(A, x0)


BƯỚC 1: Tìm trị riêng cho ma trận


Matrix([
[5.0,    0,   0,   0,   0],
[  0, -5.0,   0,   0,   0],
[  0,    0, 3.0,   0,   0],
[  0,    0,   0, 2.0,   0],
[  0,    0,   0,   0, 1.0]])

--- Rơi vào TH2 ---
 k   y2k_1   y2k_2  y2k_3  y2k_4  y2k_5   theta   v1_1   v1_2   v1_3   v1_4   v1_5   v2_1   v2_2   v2_3   v2_4   v2_5
 1  0.7071  0.7071 0.0000 0.0000 0.0000 25.0000 1.0000 0.0000 0.0000 0.0000 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000
 2 17.6777 17.6777 0.0000 0.0000 0.0000 25.0000 1.0000 0.0000 0.0000 0.0000 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000

BƯỚC 3: Tìm trị riêng cho ma trận


Matrix([
[0, 0,   0,   0,   0],
[0, 0,   0,   0,   0],
[0, 0, 3.0,   0,   0],
[0, 0,   0, 2.0,   0],
[0, 0,   0,   0, 1.0]])

--- Rơi vào TH1 ---
 k     y_1     y_2    y_3    y_4    y_5  lambda     x_1     x_2    x_3    x_4    x_5
 1 -0.0000 -0.0000 1.3416 0.8944 0.4472 -0.0000 -0.0000 -0.0000 0.8018 0.5345 0.2673
 2 -0.0000 -0.0000 2.4054 1.0690 0.2673  3.0000 -0.0000 -0.0000 0.9091 0.4041 0.1010
 3 -0.0000 -0.0000 2.7274 0.8081 0.1010  3.0000 -0.0000 -0.0000 0.9582 0.2839 0.0355
 4 -0.0000 -0.0000 2.8746 0.5678 0.0355  3.0000 -0.0000 -0.0000 0.9810 0.1938 0.0121
 5 -0.0000 -0.0000 2.9429 0.3875 0.0121  3.0000 -0.0000 -0.0000 0.9914 0.1306 0.0041
 6 -0.0000 -0.0000 2.9743 0.2611 0.0041  3.0000 -0.0000 -0.0000 0.9962 0.0875 0.0014
 7 -0.0000 -0.0000 2.9885 0.1749 0.0014  3.0000 -0.0000 -0.0000 0.9983 0.0584 0.0005
 8 -0.0000 -0.0000 2.9949 0.1169 0.0005  3.0000 -0.0000 -0.0000 0.9992 0.0390 0.0002
 9 -0.0000 -0.0000 2.9977 0.0780 0.0002  3.0000 -0.0000 -0.0000 0.9997 0.0260 0.0001
10 -0.0000 -0.0000 2.9990 0.0520 0.0001  3.0000 -0.0000 -0.0000 0.9998 0.0173 0.0000
11 -0.0000 -0.0000 2.9995 0.0347 0.0000  3.00

...

Matrix([
[  0],
[  0],
[1.0],
[  0],
[  0]])

λ_4 = 2.0000
v_4:


Matrix([
[  0],
[  0],
[  0],
[1.0],
[  0]])

λ_5 = 1.0000
v_5:


Matrix([
[  0],
[  0],
[  0],
[  0],
[1.0]])